# Interactive IQ sweep fitter
Fit IQ loops across a parameter sweep (e.g. power, temperature, or field) for every resonator interactively.  
Results are saved to a zarr file.

In [ ]:
import numpy as np
import zarr
from citkid.pipeline.framework import plStep
from citkid.pipeline.interactive import run_sweep_fitter

# ── Load data ────────────────────────────────────────────────────────────────
# Open your raw data zarr group (read-only).
root = zarr.open(
    '',   # <-- path to raw data zarr
    mode='r'
)

# Open (or create) an output zarr group where fit results will be saved.
# Each sweep index is written to its own subgroup: sweep_000, sweep_001, …
root_out = zarr.open(
    '',   # <-- path to output zarr (will be created if it does not exist)
    mode='a'
)

# ── Sweep metadata ───────────────────────────────────────────────────────────
# n_sweep: number of sweep points (e.g. number of power levels).
# nrows:   number of resonators per sweep point.
n_sweep = ...   # int
nrows   = ...   # int

In [ ]:
# ── Define custom pipeline steps ─────────────────────────────────────────────
# make_custom_steps(sweep_idx) must return a list of plStep objects that load
# all data needed by the IQ analysis pipeline for a given sweep index.
#
# Required outputs across all steps:
#   global      : fres_all, qres_all, nrows
#   global-res  : fres, qres, ares, res_idxs
#   per-row     : ff, zf   (fine/target sweep — complex S21 vs frequency)
#   per-row     : fg, zg   (gain sweep — complex S21 vs frequency)
#
# Each plStep is constructed as:
#   plStep(name, func, arg_names, return_names, func_type)
# where func_type is one of 'global', 'global-res', 'per-row', 'vectorized'.

def make_custom_steps(sweep_idx):
    # ------------------------------------------------------------------ #
    # Replace the function bodies below with your own data loading logic. #
    # ------------------------------------------------------------------ #

    def load_global_data():
        # fres_all (N_total,): full frequency list across all resonators (Hz)
        # qres_all (N_total,): corresponding Q-factor estimates
        # nrows (int):         number of resonators in this analysis
        fres_all = ...
        qres_all = ...
        return fres_all, qres_all, nrows

    def load_global_res_data():
        # fres (nrows,):     resonance frequencies for this sweep point (Hz)
        # qres (nrows,):     Q-factor estimates
        # ares (nrows,):     sweep parameter value per resonator
        #                    (e.g. drive power in uW or dBm)
        # res_idxs (nrows,): integer indices mapping resonators into fres_all
        fres     = ...
        qres     = ...
        ares     = ...
        res_idxs = ...
        return fres, qres, ares, res_idxs

    def load_data_f(data_idx):
        # ff (M,): fine-sweep frequencies in Hz  (must be sorted ascending)
        # zf (M,): corresponding complex S21
        ff = ...
        zf = ...
        return ff, zf

    def load_data_g(data_idx):
        # fg (L,): gain-sweep frequencies in Hz  (must be sorted ascending)
        # zg (L,): corresponding complex S21
        fg = ...
        zg = ...
        return fg, zg

    custom_steps = [
        ('load_global_data',     load_global_data,     [],           ['fres_all', 'qres_all', 'nrows'],    'global'),
        ('load_global_res_data', load_global_res_data, [],           ['fres', 'qres', 'ares', 'res_idxs'], 'global-res'),
        ('load_data_f',          load_data_f,          ['data_idx'], ['ff', 'zf'],                         'per-row'),
        ('load_data_g',          load_data_g,          ['data_idx'], ['fg', 'zg'],                         'per-row'),
    ]
    return [plStep(*cs) for cs in custom_steps]

In [ ]:
run_sweep_fitter(
    make_custom_steps=make_custom_steps,
    cal_yaml_path='iq',       # calibration YAML alias or path
    analysis_yaml_path='iq',  # analysis YAML alias or path
    root=root_out,
    n_sweep=n_sweep,
    # x_param_name: DS attribute to use as x on the sweep scatter plot.
    # Must be a per-resonator scalar available after the pipeline runs
    # (e.g. 'ares' for drive power, 'fres' for resonance frequency).
    x_param_name='ares',
    x_name='Power (uW)',      # x-axis label shown in the scatter plot
    # y_param_name: fit result to plot on the y-axis of the scatter.
    # One of: 'fr', 'Qr', 'Qc', 'Qi', 'amp', 'phi', 'a'
    y_param_name='a',
    start_data_idx=0,         # resonator index to open on startup
)